## 0. Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

## 1. Residual Block

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU()

        if(in_channels != out_channels or stride != 1):
            self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, padding=0, bias=False)
        else:
            self.projection = None

    def forward(self, x):
        identity = x
        if self.projection is not None:
            identity = self.projection(identity)

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        x = x + identity

        x = self.bn2(x)

        return self.relu(x)

## 2. Model Definition

In [ ]:
class ResNet(nn.Module):
    def __init__(self, in_channels, out_channels, classes):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        self.block1 = nn.Sequential(
            ResidualBlock(out_channels, out_channels, stride=1),
            ResidualBlock(out_channels, out_channels, stride=1)
            )
        self.block2 = nn.Sequential(
            ResidualBlock(out_channels, 2*out_channels, stride=2),
            ResidualBlock(2*out_channels, 2*out_channels, stride=1)
            )
        self.block3 = nn.Sequential(
            ResidualBlock(2*out_channels, 4*out_channels, stride=2),
            ResidualBlock(4*out_channels, 4*out_channels, stride=1)
            )
        self.block4 = nn.Sequential(
            ResidualBlock(4*out_channels, 8*out_channels, stride=2),
            ResidualBlock(8*out_channels, 8*out_channels, stride=1)
            )

        self.avg_pooling = nn.AvgPool2d(1,1)
        self.linear = nn.Linear(512, classes)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.avg_pooling(x)
        x = x[:,:,0,0]
        return self.linear(x)


## 3. Utility Functions

In [ ]:
def eval_step(model, test_loader, criterion, device):
    model.eval()

    total = 0
    correct = 0
    test_loss = 0

    with torch.no_grad():
        for (images, labels) in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            logits = model(images)

            loss = criterion(logits, labels)

            test_loss += loss.item() * images.size(0)

            total += labels.size(0)

            preds = torch.argmax(logits, dim=1)

            correct += torch.eq(preds, labels).sum().item()

    test_loss = test_loss / total
    test_acc = correct / total * 100

    return test_loss, test_acc

def load_data(batch_size = 32, download=False):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=0, std=1)
    ])
    train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=download)
    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=download)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2, persistent_workers=True)

    return train_loader,test_loader

## 4. Training

In [ ]:
EPOCHS = 8
train_loader, test_loader = load_data(32)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = ResNet(3, 64, classes=10).to(device)
criterion = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(params=model.parameters(), lr=0.00011)


for epoch in range(EPOCHS):
    model.train()
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optim.zero_grad()

        logits = model(images)

        loss = criterion(logits, labels)

        loss.backward()

        optim.step()

        if batch_idx % 100 == 0:
            print(f"Batch: {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    test_loss, test_acc = eval_step(model, test_loader, criterion, device)
    print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

torch.save(model.state_dict(), 'model.pth')